# 15 — Bài báo, bước 2b: flat LLM, LLM few-shot phẳng bốn lớp (máy local, API)

**Phạm vi**

Một lần gọi LLM gán thẳng một trong bốn lớp BUG / FEATURE / UX / OTHER, để so với cascade LLM hai lần gọi. Thiết kế đã
pre-register ở amendment v3.3.

**Prompt**

Ghép cơ học từ hai prompt tầng v1 đã khóa; ví dụ là hợp của hai bộ ví dụ đã khóa (14 review Train). Model, temperature,
seed API, JSON schema và cách thử lại giống hệt chiến lược C.

**Kiểm soát chi phí**

Chạy thử 20 review Validation trước. Toàn bộ Validation (`RUN_VALIDATION`) và ba tập khóa (`RUN_LOCKED`) chỉ chạy khi người
nghiên cứu đã xác nhận. Mọi lần gọi được ghi log JSONL không chứa văn bản review; chạy lại notebook thì đọc từ log, không gọi lại API.

In [1]:
RUN_VALIDATION = True     # chỉ bật sau khi người nghiên cứu xác nhận chi phí
RUN_LOCKED = True         # chỉ bật sau khi prompt đã khóa và người nghiên cứu xác nhận chi phí

import os, sys, json, datetime, subprocess
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification import evaluate as E
from src.classification.strategy_llm_fewshot import prompts as P
from src.classification.strategy_llm_fewshot.classify import classify, cost_summary, results_from_log
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
train_df = pd.read_csv(os.path.join(DATA, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA, 'shopee_validation.csv'), encoding='utf-8-sig')
assert not set(train_df['review_id']) & set(val_df['review_id'])
print('Ngày chạy:', RUN_DATE, '| commit:', COMMIT, '| RUN_VALIDATION:', RUN_VALIDATION, '| RUN_LOCKED:', RUN_LOCKED)

Ngày chạy: 15/11/2026 | commit: 2927f88af499d702a5bd8145977e8f06411aca57 | RUN_VALIDATION: True | RUN_LOCKED: True


## Kiểm tra API

**Nội dung**

Một lần gọi ngắn với snapshot đã khóa; model trả về phải trùng snapshot. Key đọc từ `.env` và không bao giờ được in ra.

In [2]:
from src.utils.llm import check_api, get_client, prices
r = check_api()
assert r['ok'], f"API lỗi: {r.get('error')} — {r.get('hint', '')}"
print('Model trả về:', r['response_model'], '| khớp snapshot:', r['snapshot_match'], '| temperature', r['temperature'])
assert r['snapshot_match'], 'Model trả về khác snapshot đã khóa'
PRICES = prices()
print('Giá / 1M token:', PRICES)
client, SETTINGS = get_client()

Model trả về: gpt-4o-mini-2024-07-18 | khớp snapshot: True | temperature 0.0
Giá / 1M token: {'input': 0.15, 'output': 0.6, 'source': 'env'}


## Prompt phẳng

**Ghép prompt**

`build_flat_system` ghép hai prompt tầng v1; nếu `flat_system_v1.txt` đã có thì nội dung phải trùng kết quả ghép. Manifest ví dụ
chỉ lưu review_id, nhãn và hash văn bản.

**Kiểm tra prompt tầng**

Hash hai prompt tầng v1 phải khớp registry, bảo đảm phần mở rộng `flat` không đổi gì ở chiến lược C.

In [3]:
REG = open(P.REGISTRY, encoding='utf-8').read()
for t in (1, 2):
    assert P.load_prompt(t, 'v1', train_df).hash(SETTINGS) in REG, f'hash prompt tầng {t} không khớp registry'
print('Hash prompt tầng 1 và tầng 2 khớp registry.')
VERSION = 'v1'
flat_text = P.build_flat_system(open(P.system_path(1, 'v1'), encoding='utf-8').read(), open(P.system_path(2, 'v1'), encoding='utf-8').read())
SYS_PATH = P.system_path('flat', VERSION)
if os.path.exists(SYS_PATH):
    assert open(SYS_PATH, encoding='utf-8').read().strip() == flat_text.strip(), 'flat_system_v1.txt khác kết quả ghép'
else:
    open(SYS_PATH, 'w', encoding='utf-8').write(flat_text + '\n')
P.write_manifest(P.flat_exemplars(train_df, 'v1'), 'flat', VERSION, rule=P.FLAT_RULE)
PROMPT = P.load_prompt('flat', VERSION, train_df)
FLAT_HASH = PROMPT.hash(SETTINGS)
print(f'prompt_hash {FLAT_HASH[:16]}… (chưa khóa) | {len(PROMPT.exemplars)} ví dụ:', ', '.join(e['label'] for e in PROMPT.exemplars))
print('Độ dài system prompt:', len(PROMPT.system), 'ký tự')

Hash prompt tầng 1 và tầng 2 khớp registry.
prompt_hash 62baaf4f076a88ad… (chưa khóa) | 14 ví dụ: BUG, FEATURE, UX, OTHER, BUG, FEATURE, UX, OTHER, BUG, FEATURE, UX, OTHER, UX, OTHER
Độ dài system prompt: 3894 ký tự


## Chạy thử 20 review Validation

**Mẫu**

10 review actionable và 10 review OTHER của Validation, lấy bằng `random_state=42` như lần chạy thử của GĐ8, nên so được với
nhãn cascade LLM của cùng 20 review trong `llm_pilot_calls.jsonl` mà không gọi thêm API.

**Ước lượng chi phí**

Chi phí thực của 20 lần gọi được ngoại suy cho 350 review Validation và 1.650 review của ba tập khóa.

In [4]:
is_act = val_df['gold_label'] != 'OTHER'
pilot = pd.concat([val_df[is_act].sample(10, random_state=42), val_df[~is_act].sample(10, random_state=42)])
PILOT_LOG = os.path.join(RES, 'paper_flat_llm_pilot_calls.jsonl')
if os.path.exists(PILOT_LOG):
    print('Đã chạy thử trước đó — đọc lại từ log (cột giây không có trong log).')
    pf = results_from_log(PILOT_LOG, 'flat')
else:
    pf = classify(pilot, PROMPT, client, SETTINGS, PILOT_LOG)
c1, c2 = results_from_log(os.path.join(RES, 'llm_pilot_calls.jsonl'), 1), results_from_log(os.path.join(RES, 'llm_pilot_calls.jsonl'), 2)
cas = c1[['review_id', 'pred_label']].merge(c2[['review_id', 'pred_label']], on='review_id', how='left', suffixes=('_t1', '_t2'))
cas['cascade LLM'] = np.where(cas['pred_label_t1'] == 'ACTIONABLE', cas['pred_label_t2'], 'OTHER')
show = pilot[['review_id', 'gold_label']].merge(pf[['review_id', 'pred_label']].rename(columns={'pred_label': 'flat LLM'}), on='review_id') \
    .merge(cas[['review_id', 'cascade LLM']], on='review_id', how='left')
print(show.assign(review_id=lambda d: d['review_id'].str[:8]).to_string(index=False))
print(f"\nĐúng nhãn bốn lớp: flat {(show['flat LLM'] == show['gold_label']).sum()}/20 | cascade {(show['cascade LLM'] == show['gold_label']).sum()}/20")
PC = pd.DataFrame([cost_summary(pf, PRICES, 'flat – chạy thử')])
print(PC.round(5).to_string(index=False))
per = PC.loc[0, 'usd_/1000_review'] / 1000
print(f'Ước lượng: Validation 350 ≈ ${per * 350:.3f} | Hold-out + focused + Lazada 1.650 ≈ ${per * 1650:.3f} '
      '(giới hạn trên, tính token cached theo giá đầy đủ)')

Đã chạy thử trước đó — đọc lại từ log (cột giây không có trong log).
review_id gold_label flat LLM cascade LLM
 7eb7cc1e        BUG      BUG         BUG
 01682bdb         UX       UX          UX
 e8e1bc81         UX       UX          UX
 0bbc94d6         UX       UX          UX
 ff9ece08         UX       UX          UX
 6f55e849        BUG      BUG         BUG
 41abaa02         UX       UX          UX
 986e1798         UX       UX          UX
 e661c118        BUG      BUG         BUG
 44364696         UX    OTHER          UX
 1d7faffe      OTHER    OTHER       OTHER
 6becb565      OTHER    OTHER       OTHER
 bb0a1292      OTHER    OTHER       OTHER
 7922ac51      OTHER    OTHER       OTHER
 e66142e1      OTHER    OTHER       OTHER
 da5796a1      OTHER    OTHER       OTHER
 504a3ed9      OTHER    OTHER       OTHER
 bfa18d81      OTHER    OTHER       OTHER
 eb565b64      OTHER    OTHER       OTHER
 7cb93df3      OTHER    OTHER       OTHER

Đúng nhãn bốn lớp: flat 19/20 | cascade 20/20
  

## Toàn bộ Validation

**Điều kiện chạy**

Chỉ chạy khi `RUN_VALIDATION = True`. Kết quả gồm Macro-F1 bốn lớp, F1 từng lớp và recall ACTIONABLE suy ra (BUG/FEATURE/UX
gộp lại), đặt cạnh cascade LLM trên Validation dựng từ log GĐ8.

In [5]:
VAL_LOG = os.path.join(RES, 'paper_flat_llm_val_calls.jsonl')
if RUN_VALIDATION or os.path.exists(VAL_LOG):
    if os.path.exists(VAL_LOG):
        done = results_from_log(VAL_LOG, 'flat')
        todo = val_df[~val_df['review_id'].isin(done['review_id'])]
        print(f'Log có {len(done)} review; còn {len(todo)} review chưa gọi.')
    else:
        done, todo = pd.DataFrame(), val_df
    if len(todo):
        assert RUN_VALIDATION, 'Còn review chưa gọi nhưng RUN_VALIDATION = False'
        new = classify(todo, PROMPT, client, SETTINGS, VAL_LOG, progress_every=50)
        done = pd.concat([done, new], ignore_index=True)
    VF = val_df[['review_id', 'gold_label']].merge(done, on='review_id', how='left')
    assert VF['pred_label'].notna().all()
    LBL = {l: i for i, l in enumerate(E.FOUR_CLASSES)}
    y, p = VF['gold_label'].map(LBL).to_numpy(), VF['pred_label'].map(LBL).fillna(-1).astype(int).to_numpy()
    m = E.metrics(y, p, E.FOUR_CLASSES)
    act_g, act_p = y != LBL['OTHER'], p != LBL['OTHER']
    rec_act = (act_g & act_p).sum() / act_g.sum(); prec_act = (act_g & act_p).sum() / max(act_p.sum(), 1)
    # cascade LLM trên Validation từ log GĐ8 (không gọi API)
    L = os.path.join(RES, 'llm_calls.jsonl')
    t1, t2 = results_from_log(L, 1), results_from_log(L, 2)
    cv = val_df[['review_id', 'gold_label']].merge(t1[['review_id', 'pred_label']], on='review_id', how='left') \
        .merge(t2[['review_id', 'pred_label']], on='review_id', how='left', suffixes=('_t1', '_t2'))
    assert cv['pred_label_t1'].notna().all(), 'log GĐ8 thiếu tầng 1 cho Validation'
    cv['pred'] = np.where(cv['pred_label_t1'] == 'ACTIONABLE', cv['pred_label_t2'], 'OTHER')
    assert cv['pred'].notna().all(), 'log GĐ8 thiếu tầng 2 cho review tầng 1 giữ lại'
    pc = cv['pred'].map(LBL).to_numpy()
    mc = E.metrics(y, pc, E.FOUR_CLASSES)
    rec_c = ((pc != LBL['OTHER']) & act_g).sum() / act_g.sum(); prec_c = ((pc != LBL['OTHER']) & act_g).sum() / max((pc != LBL['OTHER']).sum(), 1)
    VCMP = pd.DataFrame([{'cấu hình': n, 'Macro-F1 (4 lớp)': mm['macro_f1'] * 100, **{f'F1 {c}': mm[f'f1_{c}'] * 100 for c in E.FOUR_CLASSES},
                          'Recall ACTIONABLE': ra * 100, 'Precision ACTIONABLE': pa * 100}
                         for n, mm, ra, pa in (('Flat LLM', m, rec_act, prec_act), ('Cascade LLM (GĐ8)', mc, rec_c, prec_c))]).round(1)
    print('Validation (n = 350; %):'); print(VCMP.to_string(index=False))
    print('\nMa trận nhầm lẫn flat LLM:'); print(E.confusion(y, np.where(p < 0, 3, p), E.FOUR_CLASSES).to_string())
    VC = pd.DataFrame([cost_summary(done, PRICES, 'flat – Validation')])
    print(VC.round(5).to_string(index=False))
    print('Lỗi định dạng (fallback):', int(done['fallback'].sum()), '| thử lại do parse:', int((done['attempts'] > 1).sum()))
else:
    print('Chưa chạy toàn bộ Validation (RUN_VALIDATION = False).')

Log có 350 review; còn 0 review chưa gọi.


Validation (n = 350; %):
         cấu hình  Macro-F1 (4 lớp)  F1 BUG  F1 FEATURE  F1 UX  F1 OTHER  Recall ACTIONABLE  Precision ACTIONABLE
         Flat LLM              76.1    62.7        65.1   81.8      94.9               88.7                  75.9
Cascade LLM (GĐ8)              74.2    58.1        60.0   83.3      95.3               87.3                  78.5

Ma trận nhầm lẫn flat LLM:
              đoán BUG  đoán FEATURE  đoán UX  đoán OTHER
gold BUG            21             1        2           3
gold FEATURE         3            14        2           3
gold UX              0             2       18           2
gold OTHER          16             4        0         259
             bước  n_review  n_lần_gọi  thử_lại_do_parse  fallback  lỗi_API_tạm_thời  token_vào  token_vào_cached  token_ra  chi_phí_usd  giây  giây_/1000_review  usd_/1000_review
flat – Validation       350        350                 0         0                 0     614469            551808      1750      0.0932

## Khóa prompt và dự đoán các tập khóa

**Điều kiện chạy**

Chỉ chạy khi `RUN_LOCKED = True`, sau khi Validation đã chạy xong. Prompt được khóa vào `prompts_registry.md` trước, rồi dự đoán
Hold-out, focused và Lazada từ file text-only. Nhãn gold của ba tập không được đọc; việc chấm làm trong notebook đánh giá chung.

In [6]:
if RUN_LOCKED:
    assert os.path.exists(VAL_LOG), 'Chưa có kết quả Validation'
    h, status = P.register(PROMPT, SETTINGS, RUN_DATE)
    print(f'prompt_hash {h[:16]}… ({status})')
    for s in ('holdout', 'focused', 'lazada'):
        d = pd.read_csv(os.path.join(DATA, f'{s}_text_only.csv'), encoding='utf-8-sig')
        assert list(d.columns) == ['review_id', 'text']
        log = os.path.join(RES, f'paper_flat_llm_{s}_calls.jsonl')
        done = results_from_log(log, 'flat') if os.path.exists(log) else pd.DataFrame(columns=['review_id'])
        todo = d[~d['review_id'].isin(done['review_id'])]
        new = classify(todo, PROMPT, client, SETTINGS, log, progress_every=100) if len(todo) else pd.DataFrame()
        out = pd.concat([done, new], ignore_index=True) if len(new) else done
        out = d[['review_id']].merge(out, on='review_id', how='left')
        assert out['pred_label'].notna().all()
        out.to_csv(os.path.join(RES, f'paper_flat_llm_{s}.csv'), index=False, encoding='utf-8-sig')
        print(s, len(out), cost_summary(out, PRICES, s)['chi_phí_usd'])
else:
    print('Chưa khóa prompt và chưa dự đoán các tập khóa (RUN_LOCKED = False).')

prompt_hash 62baaf4f076a88ad… (đã đăng ký (khóa))


[tierflat] 100/1000


[tierflat] 200/1000


[tierflat] 300/1000


[tierflat] 400/1000


[tierflat] 500/1000


... (8 outputs left out)

[tierflat] 200/500


[tierflat] 300/500


[tierflat] 400/500


[tierflat] 500/500
lazada 500 0.1337547


## Nhận xét

**Định dạng đầu ra**

Tổng cộng 2.020 lần gọi (20 chạy thử, 350 Validation, 1.650 cho ba tập khóa) không có lần thử lại do parse, không fallback và
không lỗi API tạm thời, nên prompt v1 được giữ nguyên theo amendment v3.3.

**Validation**

Flat LLM đạt Macro-F1 bốn lớp 76,1, so với 74,2 của cascade LLM dựng từ log GĐ8; recall ACTIONABLE 88,7 so với 87,3, precision
ACTIONABLE 75,9 so với 78,5. Lỗi lớn nhất là 16 review OTHER bị gán BUG. Validation chỉ có 71 review actionable nên chênh lệch
này chưa đủ để kết luận.

**Khóa prompt**

prompt_hash `62baaf4f…` được ghi vào `04_prompts/prompts_registry.md` ngày 15/11/2026, trước khi gọi các tập khóa.

**Chi phí**

Theo token thực, flat LLM tốn 0,266–0,271 USD cho 1.000 review (token cached tính theo giá đầy đủ), cao hơn cascade LLM ở GĐ10
(0,206 USD) vì prompt phẳng dài hơn và chạy trên mọi review. Thời gian khoảng 780–790 giây cho 1.000 review, thấp hơn cascade
(1.103 giây) vì chỉ có một lần gọi. Tổng chi phí của notebook khoảng 0,54 USD.

**Tái lập**

Các lần gọi đi qua ba system_fingerprint khác nhau của nhà cung cấp, khoảng 80% qua `fp_fc3bf93e48` trên Hold-out. Temperature 0
không bảo đảm tất định tuyệt đối; điều này được ghi nhận như ở GĐ10.

**Bước tiếp theo**

Hold-out, focused và Lazada đã có nhãn dự đoán nhưng chưa được chấm; việc chấm diễn ra một lần trong notebook đánh giá chung của bước 2.